# 01 Data Exploration

This notebook builds a tiny MNIST-Addition split from the real MNIST dataset and inspects the exact files that PSL consumes: image features, pair blocks, target atoms, truth atoms, and rules.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data.generator import generate_experiment_datasets
from data.loader import load_mnist_arrays
from utils.config import DatasetConfig, dataset_dir, neupsl_train_dir
from utils.io import load_json, load_psl

config = DatasetConfig(pretrain_size=20, pretrain_valid_size=8, train_size=8, valid_size=8, inference_size=8, overlap=0.0)
features, labels = load_mnist_arrays(download=True)
generate_experiment_datasets(config, features, labels)
data_dir = neupsl_train_dir(config)
shared_dir = dataset_dir(config.name)
data_dir

## Image Samples

These are real MNIST handwritten digits loaded through `torchvision.datasets.MNIST`.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, image, label in zip(axes.ravel(), features[:8], labels[:8]):
    ax.imshow(image.reshape(28, 28), cmap="gray")
    ax.set_title(f"label={label}")
    ax.axis("off")
plt.tight_layout()

## PSL File Shapes

`image-target-*` enumerates possible digit labels for each image. `image-sum-target-*` enumerates possible sums for each image pair. Truth files mark the correct sum with a final `1`.

In [ ]:
def preview(path, rows=5, dtype=str):
    values = load_psl(path, dtype=dtype)
    print(f"{path.name}: {len(values)} rows")
    for row in values[:rows]:
        print("  ", row)

preview(data_dir / "image-sum-block-train.txt", dtype=int)
preview(data_dir / "image-sum-target-train.txt", dtype=int)
preview(data_dir / "image-sum-truth-train.txt", dtype=int)
preview(data_dir / "image-target-train.txt", dtype=int)

In [ ]:
entity_rows = load_psl(data_dir / "entity-data-map.txt", dtype=float)
first = entity_rows[0]
print("entity-data-map columns:")
print("  entity_id + 784 normalized pixels + digit_label")
print("first entity:", int(first[0]), "feature_count:", len(first) - 2, "label:", int(first[-1]))

preview(shared_dir / "number-sum.txt", rows=8, dtype=int)
preview(shared_dir / "possible-digits.txt", rows=8, dtype=int)

## Rule Template

The rule file is a normal JSON config. Swapping `psl/rules/experiment__mnist-1.json` is the rule-library replacement point.

In [ ]:
rule_config = load_json(PROJECT_ROOT / "psl" / "rules" / "experiment__mnist-1.json")
print("Predicates:", ", ".join(rule_config["predicates"].keys()))
print("\nFirst rules:")
for rule in rule_config["rules"][:4]:
    print("-", rule)

print("\nDeep predicate options:")
print(json.dumps(rule_config["predicates"]["NeuralClassifier/2"]["options"], indent=2))